# Jane Street 滞后特征工程

## 什么是滞后特征（Lag Features）？

### 概念解释

**滞后特征**是指将过去时间点的变量值作为当前时间点的特征。在时间序列预测中，这是一个非常强大的技巧。

### 为什么需要滞后特征？

1. **时间依赖性**: 金融数据具有很强的自相关性，过去的值往往包含对未来值的预测信息
2. **惯性效应**: 市场反应往往有惯性，前一时刻的状态会影响当前时刻
3. **趋势捕捉**: 滞后特征可以帮助模型识别和利用趋势模式

### 在Jane Street竞赛中的应用

本次竞赛中，API会提供前一个交易日的所有responder值作为`lags`。我们需要：
1. 理解lags的格式
2. 在训练数据中创建相同的滞后特征
3. 确保训练和推理时的数据格式一致

### 滞后特征类型

在这个项目中，我们主要创建：
- **responder_lag_1**: 前一个时间点的responder值
- 对于每个responder（0-8），我们创建其滞后1期的特征

## 1. 环境设置

In [ ]:
# vscode modified
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置路径
# 使用相对路径
DATA_PATH = Path('../Dataset')
OUTPUT_PATH = Path('./processed_data')

# 创建输出目录
OUTPUT_PATH.mkdir(exist_ok=True, parents=True)

print(f"数据路径: {DATA_PATH}")
print(f"输出路径: {OUTPUT_PATH}")

## 2. 理解Lags数据格式

### API提供的lags格式

在Kaggle的实时推理API中，每个时间步（time_id == 0）会提供一个lags DataFrame，包含：
- 所有股票在前一个交易日的所有responder值
- 这是一个"lookback"机制，让我们能够访问历史信息

### lags.parquet的格式

让我们先检查lags.parquet的结构：

In [ ]:
# 检查lags文件结构
lags_path = DATA_PATH / 'lags.parquet'

if lags_path.exists():
    # 使用polars读取（更快）
    lags_pl = pl.read_parquet(lags_path)
    print("=== Lags数据结构 ===")
    print(f"形状: {lags_pl.shape}")
    print(f"\n列名: {lags_pl.columns}")
    print(f"\n前几行:")
    print(lags_pl.head(10))
    
    # 转换为pandas以便查看
    lags_df = lags_pl.to_pandas()
    print(f"\n数据类型:")
    print(lags_df.dtypes)
else:
    print("lags.parquet不存在，将在后续模拟创建")

## 3. 在训练数据中创建滞后特征

### 滞后特征创建原理

对于训练数据中的每一行（date_id, time_id, symbol_id），我们需要找到：
- 同一symbol_id
- 在前一天（date_id - 1）
- 在最后时间点（time_id == 967，即一天结束）
- 的responder值

### 为什么是前一天的最后时间点？

这是因为API在实时推理中会提供"前一交易日结束"的状态作为lags。这是最接近当前交易开始时的历史信息。

In [ ]:
def create_lags_optimized(df):
    """
    优化的滞后特征创建函数
    
    策略：
    1. 按symbol_id和date_id分组
    2. 获取每个组最后时间点的responder值
    3. 对date_id进行shift操作得到前一日的值
    4. 将滞后值join回原数据
    """
    responder_cols = [f'responder_{i}' for i in range(9)]
    existing_responders = [col for col in responder_cols if col in df.columns]
    
    # 步骤1: 获取每天每个symbol最后时间点的responder值
    last_values = (
        df.sort(['symbol_id', 'date_id', 'time_id'])
        .group_by(['symbol_id', 'date_id'])
        .agg([
            pl.col('time_id').last().alias('last_time_id'),
            *[pl.col(col).last().alias(f'{col}_last') for col in existing_responders]
        ])
    )
    
    # 步骤2: 创建前一日的值（对date_id进行shift）
    last_values_with_lag = (
        last_values
        .sort(['symbol_id', 'date_id'])
        .with_columns([
            pl.col(f'{col}_last').shift(1).over('symbol_id').alias(f'{col}_lag_1')
            for col in existing_responders
        ])
    )
    
    # 步骤3: 只保留需要的列进行join
    lag_cols = ['symbol_id', 'date_id'] + [f'{col}_lag_1' for col in existing_responders]
    last_values_with_lag = last_values_with_lag.select(lag_cols)
    
    # 步骤4: 将滞后值join回原数据
    df_with_lags = df.join(
        last_values_with_lag,
        on=['symbol_id', 'date_id'],
        how='left'
    )
    
    return df_with_lags

print("优化的滞后特征创建函数已定义")

## 4. 批量处理所有分区并保存

### 内存管理策略

由于数据集很大，我们需要：
1. 逐个分区处理
2. 处理完立即保存
3. 及时释放内存

### 输出格式

我们将创建两个文件：
- `training.parquet`: 用于训练的数据（带有滞后特征）
- `validation.parquet`: 用于验证的数据（带有滞后特征）

In [ ]:
# modified
import gc
from tqdm import tqdm

def process_all_partitions(
    data_path=DATA_PATH,
    output_path=OUTPUT_PATH,
    validation_dates=100,  # 最后100天用于验证
    skip_dates=500,  # 跳过前500天（可选）
    save_every=1  # 每处理几个分区保存一次
):
    """
    处理所有分区并创建滞后特征（修复版）
    
    关键修复：先处理所有分区并合并，然后全局分割训练/验证集
    
    原始数据分区结构（按date_id范围）：
    - partition_id=0: date_id 0-160
    - partition_id=1: date_id 161-325
    - ...  
    - partition_id=9: date_id 1480-1698
    
    参数:
        data_path: 输入数据路径
        output_path: 输出路径
        validation_dates: 用作验证集的天数（全局的最后N天）
        skip_dates: 跳过的开始天数
        save_every: 每处理几个分区保存一次
    """
    
    # 获取所有分区
    train_path = data_path / 'train.parquet'
    partitions = sorted([int(d.name.split('=')[1]) for d in train_path.iterdir() if d.is_dir()])
    
    print(f"找到 {len(partitions)} 个分区: {partitions}")
    print(f"配置: skip_dates={skip_dates}, validation_dates={validation_dates}")
    
    # ========== 第一步：处理所有分区，创建滞后特征 ==========
    all_data_list = []
    
    print("\n[第一步] 为所有分区创建滞后特征...")
    
    for partition_id in tqdm(partitions, desc="创建滞后特征"):
        try:
            # 读取分区数据
            file_path = train_path / f'partition_id={partition_id}' / 'part-0.parquet'
            df = pl.read_parquet(file_path)
            
            # 创建滞后特征（不过滤日期！）
            df_with_lags = create_lags_optimized(df)
            
            # 添加到列表（保留所有数据）
            all_data_list.append(df_with_lags)
            
            # 清理内存
            del df, df_with_lags
            gc.collect()
            
        except Exception as e:
            print(f"处理分区 {partition_id} 时出错: {e}")
            continue
    
    # ========== 第二步：合并所有数据 ==========
    print("\n[第二步] 合并所有分区的数据...")
    
    combined_data = pl.concat(all_data_list)
    
    # 转换float64为float32以节省空间
    for col in combined_data.columns:
        if combined_data[col].dtype == pl.Float64:
            combined_data = combined_data.with_columns([
                pl.col(col).cast(pl.Float32)
            ])
    
    print(f"合并后数据形状: {combined_data.shape}")
    print(f"日期范围: {combined_data['date_id'].min()} 到 {combined_data['date_id'].max()}")
    print(f"总天数: {combined_data['date_id'].n_unique()}")
    
    # 清理列表内存
    del all_data_list
    gc.collect()
    
    # ========== 第三步：全局过滤和分割（关键修复）==========
    print(f"\n[第三步] 全局过滤和分割...")
    print(f"  跳过date_id < {skip_dates}的数据")
    print(f"  将最后{validation_dates}天作为验证集")
    
    # 步骤3.1: 过滤掉skip_dates之前的数据
    if skip_dates > 0:
        combined_data = combined_data.filter(pl.col('date_id') >= skip_dates)
        print(f"  过滤后日期范围: {combined_data['date_id'].min()} 到 {combined_data['date_id'].max()}")
    
    # 步骤3.2: 获取所有唯一日期并排序
    all_dates = sorted(combined_data['date_id'].unique())
    print(f"  总日期数: {len(all_dates)}")
    print(f"  日期范围: {all_dates[0]} 到 {all_dates[-1]}")
    
    # 步骤3.3: 确定训练集和验证集的日期（全局分割）
    train_dates = all_dates[:-validation_dates]  # 除了最后N天的所有日期
    valid_dates = all_dates[-validation_dates:]   # 最后N天
    
    print(f"\n  训练集日期: {train_dates[0]} 到 {train_dates[-1]} ({len(train_dates)}天)")
    print(f"  验证集日期: {valid_dates[0]} 到 {valid_dates[-1]} ({len(valid_dates)}天)")
    
    # 步骤3.4: 分割数据
    training_data = combined_data.filter(pl.col('date_id').is_in(train_dates))
    validation_data = combined_data.filter(pl.col('date_id').is_in(valid_dates))
    
    print(f"\n  训练数据形状: {training_data.shape}")
    print(f"  验证数据形状: {validation_data.shape}")
    
    # 清理合并数据
    del combined_data
    gc.collect()
    
    # ========== 第四步：保存数据 ==========
    print("\n[第四步] 保存处理后的数据...")
    
    # 保存训练数据
    output_file = output_path / 'training.parquet'
    print(f"保存训练数据到: {output_file}")
    training_data.write_parquet(output_file)
    print(f"✓ 训练数据已保存")
    
    # 保存验证数据
    output_file = output_path / 'validation.parquet'
    print(f"保存验证数据到: {output_file}")
    validation_data.write_parquet(output_file)
    print(f"✓ 验证数据已保存")
    
    # 清理内存
    del training_data, validation_data
    gc.collect()
    
    # ========== 完成 ==========
    print("\n" + "="*50)
    print("处理完成!")
    print("="*50)
    print(f"输出文件:")
    print(f"- {output_path / 'training.parquet'}")
    print(f"- {output_path / 'validation.parquet'}")
    print(f"\n最终结果:")
    print(f"  训练集: date_id {train_dates[0]}-{train_dates[-1]} ({len(train_dates)}天)")
    print(f"  验证集: date_id {valid_dates[0]}-{valid_dates[-1]} ({len(valid_dates)}天)")
    print("="*50)

# 运行处理
print("开始处理所有分区...")
process_all_partitions()

## 5. 验证处理后的数据

In [ ]:
# 验证处理后的数据
training_file = OUTPUT_PATH / 'training.parquet'
validation_file = OUTPUT_PATH / 'validation.parquet'

if training_file.exists():
    print("=== 训练数据验证 ===")
    train_data = pl.read_parquet(training_file)
    print(f"形状: {train_data.shape}")
    print(f"\n列名: {train_data.columns}")
    print(f"\n滞后特征列:")
    lag_cols = [col for col in train_data.columns if 'lag' in col]
    print(lag_cols)
    
    # 检查滞后特征的缺失值
    print(f"\n滞后特征缺失值统计:")
    for col in lag_cols[:5]:  # 只显示前5个
        null_count = train_data[col].null_count()
        null_ratio = null_count / len(train_data)
        print(f"{col}: {null_count} ({null_ratio:.2%})")
    
    # 查看样本数据
    print(f"\n前几行样本:")
    sample_cols = ['date_id', 'time_id', 'symbol_id', 'responder_6', 'responder_6_lag_1']
    available_cols = [col for col in sample_cols if col in train_data.columns]
    print(train_data.select(available_cols).head(10))

if validation_file.exists():
    print("\n=== 验证数据验证 ===")
    valid_data = pl.read_parquet(validation_file)
    print(f"形状: {valid_data.shape}")
    print(f"日期范围: {valid_data['date_id'].min()} 到 {valid_data['date_id'].max()}")
    print(valid_data.select(available_cols).head(10))

## 附录

### Q1: 为什么滞后特征有缺失值？

**A**: 主要有两个原因：
1. 第一天的数据（date_id = 0）没有前一天的数据
2. 某些股票可能在某些日期没有交易数据

**处理方法**：
- 用0填充
- 用该股票的均值填充
- 用前向填充（forward fill）

### Q2: 训练和推理时的数据格式一致吗？

**A**: 应该一致！这是关键点：
- 训练时：我们自己创建滞后特征
- 推理时：API提供滞后特征
- 两者格式必须完全相同

### Q3: 可以创建多期滞后吗？

**A**: 理论上可以，但在这个竞赛中：
- API只提供前一日的数据
- 创建多期滞后在推理时无法获取对应数据
- 所以只能使用lag_1

### Q4: 滞后特征对模型性能影响大吗？

**A**: 通常影响很大！
- 时间序列数据有很强的自相关性
- 滞后特征往往是最重要的特征之一
- 社区的解决方案几乎都使用了滞后特征